# Features and Chronological Split

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data' / 'processed' / 'Charging_Data_clean.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Could not find cleaned charging data from the current directory.')
sys.path.insert(0, str(ROOT / 'team' / 'mariam_availability'))
from src.occupancy import load_sessions
from src.features import (build_features, split_by_time, FeatureEncoder,
                          TRAIN_END, VAL_END, HIST_WINDOW, FORBIDDEN)
np.random.seed(42)

### Build start time features

In [ ]:
sessions = load_sessions(ROOT / "data" / "processed" / "Charging_Data_clean.csv")
f = build_features(sessions)
feature_names = [c for c in f.columns if c not in {"start", "target_min", "is_abnormal"}]
print(f"FEATURE TABLE\nRows            : {len(f):,}\nFeature columns : {len(feature_names)}\nFeature names   : {feature_names}")
f.head()

Calendar, weather, prior charger history, and site activity are calculated at session start. Outcome columns remain for splitting and scoring.

### Chronological split

In [ ]:
train, val, test = split_by_time(f)
for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:12s}: {len(part):>8,} | {part.start.min()} to {part.start.max()} | abnormal share {part.is_abnormal.mean():.4f}")

The three date ranges are strictly ordered. Automated tests check the exact split boundaries.

### Training subset and encoder

In [ ]:
train_n = train.loc[train.is_abnormal == 0].copy()
CAP = float(train_n.target_min.quantile(.99))
encoder = FeatureEncoder().fit(train_n)
encoded_names = encoder.feature_names
state = encoder.get_state()
print(f"TRAIN ENCODER\nTrain normal rows : {len(train_n):,}\nEncoded features  : {len(encoded_names)}\nCAP (minutes)     : {CAP:.3f}")

The category maps and numeric medians are fit using normal training sessions only. Automated tests cover unseen categories and missing-value filling across all splits.

### Write shared artifacts

In [ ]:
ART=ROOT/"team"/"mariam_availability"/"artifacts"; ART.mkdir(parents=True,exist_ok=True)
for name,obj in [("features_all.pkl",f),("encoder_state.pkl",state),("config_split.pkl",{"TRAIN_END":TRAIN_END,"VAL_END":VAL_END,"HIST_WINDOW":HIST_WINDOW,"CAP":CAP})]:
    path=ART/name
    with path.open("wb") as out: pd.to_pickle(obj,out)
    print(f"Wrote {path} ({path.stat().st_size:,} bytes)")

The full feature table and train-only encoder state are saved for this notebook.

## Final Summary

In [ ]:
print(f"Split sizes : {len(train):,} / {len(val):,} / {len(test):,}")
print(f"Features ({len(encoded_names)}): {encoded_names}"); print(f"CAP         : {CAP:.3f}")
print(f"Artifacts   : {ART}"); 